# Workshop 2

| Minutes | Activity |
|---|---|
| 0–8 | Recap of workshop 1, this time on real data |
| 8–20 | Code-along: your first model from scratch |

Save your own copy in Colab (**File → Save a copy in Drive**). Click the **play button** beside a code cell to run it. Work from top to bottom. Cells with sliders redraw when you drag.

## Recap 1: Real data from Hugging Face (2 minutes)

Last time the cars were made up. Today they are real. [Hugging Face](https://huggingface.co/datasets) hosts thousands of public datasets and models, and `load_dataset` downloads one by name.

We use [`scikit-learn/auto-mpg`](https://huggingface.co/datasets/scikit-learn/auto-mpg): 398 cars from 1970 to 1982 with their fuel economy in miles per gallon (`mpg`). More powerful cars burn more fuel, which is a pattern a model can learn.

Run the cell. It downloads the data and shows the first five cars.

In [ ]:
%pip install -q datasets  # The Hugging Face library. Colab usually has it already.

import matplotlib.pyplot as plt  # Draws graphs.
import numpy as np  # Does maths on many numbers at once.
import pandas as pd  # Stores data in tables.
from datasets import load_dataset  # Downloads datasets from Hugging Face.
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact  # Sliders and menus.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures

cars = load_dataset("scikit-learn/auto-mpg")["train"].to_pandas()  # Download, then turn into a table.
cars.head()  # Show the first five rows.

### What is a DataFrame?

`cars` is a **DataFrame**: a pandas table, like a spreadsheet you control with code. Each **row** is one example (a car). Each **column** is one measurement (`mpg`, `weight`, …).

| Code | What it does |
|---|---|
| `len(cars)` | Counts the rows |
| `cars["mpg"]` | Selects one column: a list of numbers with a name |
| `cars[["hp"]]` | Selects a column as a one-column table. Models take tables as input, because there can be several input columns |
| `cars["horsepower"] / 100` | Divides every value in the column at once |
| `cars["hp"] = …` | Adds a new column |

Real data is messy. `horsepower` is missing for 6 cars, marked with a `?`, so pandas read the whole column as text. The next cell turns it into numbers, drops the cars with missing values, and adds a column with horsepower in hundreds, so the numbers stay small.

In [ ]:
cars["horsepower"] = pd.to_numeric(cars["horsepower"], errors="coerce")  # Text to numbers. "?" becomes missing (NaN).
cars = cars.dropna()  # Drop the rows with a missing value.
cars["hp"] = cars["horsepower"] / 100  # A new column: horsepower in hundreds.

print(len(cars), "cars")
cars.plot.scatter(x="hp", y="mpg", s=5, xlabel="Horsepower (hundreds)", ylabel="Miles per gallon")  # Plot straight from the table.
plt.show()

## Recap 2: A model is a rule with parameters. Loss measures how wrong it is (1.5 minutes)

Model: $\text{mpg} = \text{intercept} + \text{slope} \times x$, where $x$ is hundreds of horsepower above 100. Two parameters.

Loss: $\text{MSE} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2$, the average squared error.

Drag the sliders. On the left, the line and the cars. On the right, the loss for **every** slope and intercept: a bowl. The dot is where you are. Find the bottom. It is near slope -16, intercept 24, with a loss of about 24.

In [ ]:
x = cars["hp"] - 1  # Hundreds of horsepower above 100. Centring on zero keeps training well behaved.
y = cars["mpg"]


def loss(slope, intercept):
    return np.mean((y - (slope * x + intercept)) ** 2)


slopes, intercepts = np.meshgrid(np.linspace(-30, 10, 60), np.linspace(0, 50, 60))  # A grid of parameter pairs.
loss_surface = np.vectorize(loss)(slopes, intercepts)  # The loss at every pair.


def draw_fit(slope, intercept, path=None):
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.scatter(x, y, s=5)
    left.plot([-0.6, 1.4], [intercept - 0.6 * slope, intercept + 1.4 * slope], color="black")
    left.set(xlabel="Hundreds of hp above 100", ylabel="Miles per gallon", ylim=(0, 50))
    left.set_title(f"mpg = {intercept:.1f} + {slope:.1f} × x.   Loss: {loss(slope, intercept):,.0f}")
    right.contour(slopes, intercepts, loss_surface, levels=np.geomspace(25, 1500, 12))
    if path is not None:
        right.plot(path[:, 0], path[:, 1], color="tab:orange", linewidth=1, marker="o", markersize=3)
    right.scatter([slope], [intercept], color="tab:orange", s=100, zorder=3)
    right.set(xlabel="Slope", ylabel="Intercept", xlim=(-30, 10), ylim=(0, 50), title="Loss for every slope and intercept")
    plt.show()


@interact(slope=FloatSlider(min=-30, max=10, step=0.5, value=0), intercept=FloatSlider(min=0, max=50, step=1, value=10))
def explore(slope, intercept):
    draw_fit(slope, intercept)

## Recap 3: Training walks downhill (1.5 minutes)

The **gradient** is the slope of the bowl at the current parameters. Backpropagation calculates it for every parameter. Each step moves against it:

$\text{parameter} \leftarrow \text{parameter} - \text{learning rate} \times \text{gradient}$

In [ ]:
def gradient_descent(steps, learning_rate):
    slope, intercept = 0.0, 10.0  # Start from a guess.
    path = [(slope, intercept)]
    for step in range(steps):
        error = slope * x + intercept - y  # Forward pass, then compare with the answers.
        slope = slope - learning_rate * 2 * np.mean(error * x)  # Step downhill.
        intercept = intercept - learning_rate * 2 * np.mean(error)
        path.append((slope, intercept))
    return np.array(path)


@interact(
    steps=IntSlider(min=0, max=20, value=0),
    learning_rate=Dropdown(options={"Small (0.1)": 0.1, "Large (0.5)": 0.5, "Too large (1.1)": 1.1}),
)
def train(steps, learning_rate):
    path = gradient_descent(steps, learning_rate)
    draw_fit(path[-1, 0], path[-1, 1], path=path)

## Recap 4: Low training loss can mislead (1.5 minutes)

Pretend we only had **13 cars**: every 30th car by horsepower. The rest are set aside.

| Code | What it does |
|---|---|
| `PolynomialFeatures(degree)` | Adds `hp²`, `hp³`, … up to `hp^degree`, so a straight-line model can bend |
| `MinMaxScaler()` | First squashes `hp` to the range 0 to 1, so the powers stay small |
| `make_pipeline(a, b, c)` | Chains the steps into one model with `fit` and `predict` |

A higher degree gives the model more parameters, so the curve can bend more. Drag the degree up and watch the training loss.

In [ ]:
by_hp = cars.sort_values("hp")
few = by_hp.iloc[15::30]  # Every 30th car by horsepower: 13 cars spread across the range.
others = by_hp.drop(few.index).sample(frac=1, random_state=0)  # The rest, shuffled.
val = others.iloc[:120]  # Validation: compare models and choose one.
test = others.iloc[120:240]  # Test: one final check.
pool = others.iloc[240:]  # Spare cars for later.
print(len(few), len(val), len(test), len(pool))


def fit_curve(degree, train=few):
    curve = make_pipeline(MinMaxScaler(), PolynomialFeatures(degree), LinearRegression())
    return curve.fit(train[["hp"]], train["mpg"])


def mse(curve, table):
    return mean_squared_error(table["mpg"], curve.predict(table[["hp"]]))


def draw_curve(curve, train=few, show_val=False, title="", ax=None):
    if ax is None:
        ax = plt.figure(figsize=(6, 3.6)).gca()
    grid = pd.DataFrame({"hp": np.linspace(0.4, 2.4, 300)})
    ax.scatter(train["hp"], train["mpg"], zorder=3, label=f"Training cars ({len(train)})")
    if show_val:
        ax.scatter(val["hp"], val["mpg"], s=8, color="tab:orange", label="Validation cars")
    ax.plot(grid["hp"], curve.predict(grid), color="black", label="Model")
    ax.set(xlabel="Horsepower (hundreds)", ylabel="Miles per gallon", ylim=(0, 50), title=title)
    ax.legend(loc="upper right")
    if ax.figure.axes == [ax]:
        plt.show()


@interact(degree=IntSlider(min=1, max=9, value=1))
def flex(degree):
    curve = fit_curve(degree)
    draw_curve(curve, title=f"Degree {degree}.   Training loss: {mse(curve, few):.1f}")

## Recap 5: Validation loss reveals overfitting (1.5 minutes)

The model never trains on the validation cars, so they act like new cars.

In [ ]:
degrees = range(1, 10)
train_by_degree = [mse(fit_curve(d), few) for d in degrees]
val_by_degree = [mse(fit_curve(d), val) for d in degrees]


@interact(degree=IntSlider(min=1, max=9, value=1))
def validate(degree):
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    curve = fit_curve(degree)
    draw_curve(curve, show_val=True, ax=left, title=f"Degree {degree}: training {mse(curve, few):.1f}, validation {mse(curve, val):,.0f}")
    right.plot(degrees, train_by_degree, marker="o", label="Training")
    right.plot(degrees, val_by_degree, marker="o", label="Validation")
    right.axvline(degree, color="black", linewidth=1, linestyle="--")
    right.set(xlabel="Degree", ylabel="Loss (MSE, log scale)", yscale="log")
    right.legend()
    plt.show()

## Code-along: your first model from scratch (12 minutes)

244 restaurant bills and the tip left on each. Does a bigger bill mean a bigger tip?

### 1. Load

[`scikit-learn/tips`](https://huggingface.co/datasets/scikit-learn/tips) on Hugging Face. Hold out 30% of the bills for the check.

### 2. Look

One dot per bill: bill across, tip up.

### 3. Fit

A line: $\text{tip} = \text{intercept} + \text{slope} \times \text{bill}$.

In [ ]:

print("slope:", model.coef_[0], "intercept:", model.intercept_)

### 4. Predict

New bills go in with the same shape as the training bills.

In [ ]:
new_bills = pd.DataFrame({"total_bill": [20, 50]})


### 5. Check

Score on the held-out bills. The baseline always guesses the average tip.

In [1]:
print("model MSE:", ...)
print("baseline MSE:", mean_squared_error(test["tip"], np.full(len(test), train["tip"].mean())))

model MSE: Ellipsis


NameError: name 'mean_squared_error' is not defined

That is the whole recipe: load, look, fit, predict, check. Every model you meet from here on, up to a neural network, uses the same `fit` and `predict`.